[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/1_OD.ipynb)

# Deteccion de Objetos con RF-DETR

La deteccion de objetos es una tarea fundamental en vision artificial. Consiste en encontrar objetos en imagenes y dibujar cajas alrededor de ellos (bounding boxes) indicando que objeto es.

**Diferencia con clasificacion:**
- Clasificacion: "Esta imagen completa contiene un gato" → 1 etiqueta
- Deteccion: "Hay 3 gatos en las posiciones X, Y, Z" → multiples boxes con etiquetas

En este notebook veremos RF-DETR, un detector moderno basado en Transformers.

## Evolucion de los Detectores

### Arquitecturas Clasicas (CNN):
- **R-CNN, Fast R-CNN, Faster R-CNN**: Detectores en dos etapas (propuestas + clasificacion)
- **YOLO (You Only Look Once)**: Detector en una etapa, muy rapido
  - YOLOv3, YOLOv4, YOLOv5, YOLOv235894... (muchas versiones)
  - Ventaja: velocidad (tiempo real en video)
  - Desventaja: arquitectura compleja con muchos "trucos", LICENCIA !!

### Arquitecturas Modernas (Transformers):
- **DETR (DEtection TRansformer)**: Primer detector con Transformers
- **RF-DETR**: Version rapida y eficiente de DETR
  - Ventaja: arquitectura limpia, sin anchors ni NMS
  - Ventaja: precision SOTA (State of the Art), LICENCIAS 


## Configuracion e Imports

In [ ]:
# Instalar libreria RFDETR (roboflow DETR)
try:
    from rfdetr import RFDETRMedium
    print("RFDETR importado!")
except ImportError:
    print("Instalando rfdetr...")
    %pip install rfdetr
    from rfdetr import RFDETRMedium
import torch
torch.cuda.empty_cache()
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
from io import BytesIO
import sys
from pathlib import Path
import logging
logging.getLogger().setLevel(logging.ERROR)
import warnings
warnings.filterwarnings("ignore") # evitar warnings innecesarios

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")

## Carga del Modelo RF-DETR

RF-DETR tiene varios tamaños: l (large), m (medium), s (small), x (extra large). Usaremos `rf-detr-medium.pth` que da buen balance.

El modelo viene pre-entrenado en **COCO dataset** con **80 categorias** de objetos comunes: personas, coches, animales, muebles, electronica, etc.

In [ ]:
coco_model = RFDETRMedium()

coco_model.optimize_for_inference(compile=False)

print("Modelo RF-DETR cargado")
print(f"Clases disponibles: {len(coco_model.class_names)} categorias COCO")

## Carga de Imagenes de Ejemplo

In [ ]:
# Cargar imagenes desde GitHub
url_cars = "https://github.com/sergiovillanueva/Modelos_Fundacionales/raw/main/assets/cars.jpg"
image_cars = Image.open(BytesIO(requests.get(url_cars).content)).convert("RGB")

url_person = "https://github.com/sergiovillanueva/Modelos_Fundacionales/raw/main/assets/person_cars.jpg"
image_person = Image.open(BytesIO(requests.get(url_person).content)).convert("RGB")

url_bananas = "https://github.com/sergiovillanueva/Modelos_Fundacionales/raw/main/assets/bananas.jpg"
image_bananas = Image.open(BytesIO(requests.get(url_bananas).content)).convert("RGB")

url_fruits = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/fruits.jpg"
image_fruits = Image.open(BytesIO(requests.get(url_fruits).content)).convert("RGB")

print("Imagenes cargadas")

## Deteccion de Objetos

Vamos a detectar objetos en las imagenes. RF-DETR devolvera:
- **Bounding boxes**: coordenadas [x1, y1, x2, y2] (left, top, right, bottom)
- **Clases**: que objeto es (coche, persona, etc.)
- **Confianza**: probabilidad de 0 a 1

### Antes de dibujar: los nombres de las clases

Las versiones nuevas de `rfdetr` devuelven `class_names` como lista de 80 nombres, mientras que el `class_id` de cada deteccion es el identificador original de COCO, que llega hasta 90 porque el dataset dejo huecos al retirar categorias. Indexar la lista con ese identificador desplaza las etiquetas: una persona aparece como `bicycle` y un coche como `motorcycle`.

La solucion es no traducir a mano. El propio modelo ya adjunta el nombre correcto en `detections.data["class_name"]`, y funciona igual para el modelo COCO y para uno entrenado por ti.


In [ ]:
# Los nombres de clase, de forma fiable
#
# RF-DETR devuelve un objeto Detections de supervision. En las versiones nuevas trae ya
# el nombre en detections.data["class_name"], y conviene usarlo: en los modelos COCO el
# class_id es la categoria original del dataset (1 a 90, con huecos) y NO la posicion
# dentro de class_names, que solo tiene 80 nombres. Indexar directo desplaza las
# etiquetas y una persona acaba etiquetada como bicycle.

def nombres_de(detections, model):
    """Devuelve el nombre de clase de cada deteccion."""
    datos = getattr(detections, "data", {}) or {}
    if "class_name" in datos:
        return [str(nombre) for nombre in datos["class_name"]]

    # Respaldo por si alguna version no adjunta el nombre
    tabla = model.class_names
    if isinstance(tabla, dict):
        return [tabla.get(int(c), str(c)) for c in detections.class_id]
    try:
        from rfdetr.assets.coco_classes import COCO_CLASSES
        if list(tabla) == list(COCO_CLASSES.values()):
            return [COCO_CLASSES.get(int(c), str(c)) for c in detections.class_id]
    except ImportError:
        pass
    return [tabla[int(c)] if 0 <= int(c) < len(tabla) else str(c) for c in detections.class_id]


print("Ayuda lista. Ejemplo con la foto de coches:")
_pruebas = coco_model.predict(np.array(image_cars), threshold=0.5)
print(list(zip(_pruebas.class_id[:5], nombres_de(_pruebas, coco_model)[:5])))


In [ ]:
def detect_objects(image, model, conf_threshold=0.5):
    """Detecta objetos usando RF-DETR"""

    # Convertir PIL a numpy
    img_array = np.array(image)

    detections = model.predict(img_array, threshold=conf_threshold)
    class_names = nombres_de(detections, model)

    print(f"\nDetectados {len(detections.xyxy)} objetos:\n")

    # Dibujar detecciones
    img_result = img_array.copy()

    for box, cls, class_name, score in zip(detections.xyxy, detections.class_id, class_names, detections.confidence):
        x1, y1, x2, y2 = map(int, box)

        print(f"  {class_name}: {score:.2f} en [{x1}, {y1}, {x2}, {y2}]")

        # Dibujar box y texto
        color = ((int(cls) * 100) % 256, (int(cls) * 150) % 256, (int(cls) * 200) % 256)
        cv2.rectangle(img_result, (x1, y1), (x2, y2), color, 3)
        text = f"{class_name} {score:.2f}"
        cv2.putText(img_result, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Visualizar
    plt.figure(figsize=(12, 8))
    plt.imshow(img_result)
    plt.title(f"Detecciones (threshold={conf_threshold})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Ejemplos:
# detect_objects(image_cars, coco_model)
# detect_objects(image_person, coco_model)
# detect_objects(image_fruits, coco_model)


## Experimentar con Thresholds

El threshold de confianza controla cuantas detecciones vemos:
- **Threshold alto (0.7-0.9)**: Solo detecciones muy seguras, menos false positives (mejor precisión)
- **Threshold bajo (0.2-0.4)**: Mas detecciones, pero puede haber errores (mejor recall)

Puedes probar con diferentes valores:

In [ ]:
detect_objects(image_person, coco_model, conf_threshold=0.5)

### Lo mismo con un deslizador

Mueve el umbral y mira aparecer y desaparecer detecciones. Es la forma rapida de encontrar el punto donde no sobran cajas ni faltan objetos.


In [ ]:
from ipywidgets import interact, FloatSlider

@interact(umbral=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5, description="Umbral"))
def probar_umbral(umbral):
    detect_objects(image_person, coco_model, conf_threshold=umbral)


## Ver todas las clases COCO

Veamos que objetos puede detectar el modelo:

In [ ]:
# Mostrar todas las clases disponibles
#
# Ojo con los numeros: COCO tiene 80 clases pero sus identificadores llegan hasta el 90,
# porque quedaron huecos al retirar categorias del dataset original. Falta el 12, el 26,
# el 29... Por eso la tabla va por identificador y no por posicion.
try:
    from rfdetr.assets.coco_classes import COCO_CLASSES

    print(f"Clases COCO que RF-DETR puede detectar: {len(COCO_CLASSES)}\n")
    for idx, name in sorted(COCO_CLASSES.items()):
        print(f"{idx:2d}: {name}")

    huecos = [i for i in range(1, max(COCO_CLASSES) + 1) if i not in COCO_CLASSES]
    print(f"\nIdentificadores sin clase: {huecos}")
except ImportError:
    # Version antigua de rfdetr: class_names era un diccionario
    tabla = coco_model.class_names
    elementos = tabla.items() if isinstance(tabla, dict) else enumerate(tabla)
    print(f"Clases COCO que RF-DETR puede detectar: {len(tabla)}\n")
    for idx, name in elementos:
        print(f"{idx:2d}: {name}")


## Aplicacion Industrial: Contar Objetos

En aplicaciones industriales es comun necesitar contar objetos. Vamos a crear una funcion que cuente automaticamente.

In [ ]:
def count_objects_by_class(image, model, conf_threshold=0.5):
    """Cuenta objetos agrupados por clase"""

    img_array = np.array(image)
    detections = model.predict(img_array, threshold=conf_threshold)
    class_names = nombres_de(detections, model)

    # Contar por clase
    class_counts = {}
    for class_name in class_names:
        class_counts[class_name] = class_counts.get(class_name, 0) + 1

    print("objetos:\n")
    for class_name, count in class_counts.items():
        print(f"  {class_name}: {count}")

    # Dibujar detecciones
    img_result = img_array.copy()

    for box, class_name, score in zip(detections.xyxy, class_names, detections.confidence):
        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(img_result, (x1, y1), (x2, y2), (0, 255, 0), 3)  # RGB - Verde
        text = f"{class_name} {score:.2f}"
        cv2.putText(img_result, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Visualizar
    plt.figure(figsize=(12, 8))
    plt.imshow(img_result)
    plt.title(f"Total: {len(detections.class_id)} objetos")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    return class_counts

# Contar objetos en imagen
# counts = count_objects_by_class(image_bananas, coco_model)
# counts = count_objects_by_class(image_person, coco_model)


### Dibujar el resultado con `supervision`

`supervision` es la libreria de Roboflow para anotar resultados. Hace en tres lineas lo que arriba hicimos a mano con OpenCV, y queda mucho mejor.


In [ ]:
try:
    import supervision as sv
except ImportError:
    %pip install -q supervision
    import supervision as sv

def dibujar_bonito(image, model, conf_threshold=0.5):
    """Mismas detecciones, anotadas con supervision."""
    detections = model.predict(np.array(image), threshold=conf_threshold)
    etiquetas = [f"{nombre} {score:.2f}"
                 for nombre, score in zip(nombres_de(detections, model), detections.confidence)]

    anotada = sv.BoxAnnotator(thickness=3).annotate(np.array(image), detections)
    anotada = sv.LabelAnnotator(text_scale=0.6).annotate(anotada, detections, labels=etiquetas)

    plt.figure(figsize=(12, 8))
    plt.imshow(anotada)
    plt.axis("off")
    plt.show()
    return detections

_ = dibujar_bonito(image_cars, coco_model, conf_threshold=0.5)


## Ejercicio: Detecta objetos en tu imagen

Prueba el detector con tus propias imagenes. Puede ser:
- Foto de tu escritorio (detectara laptop, mouse, teclado, etc.)
- Foto de la calle (coches, personas, semaforos)
- Cualquier imagen con objetos de COCO

In [ ]:
# EJERCICIO: Tu imagen aqui

# Opcion 1: Desde URL
# tu_url = "https://..."  # <- URL de tu imagen
# tu_imagen = Image.open(BytesIO(requests.get(tu_url).content)).convert("RGB")

# Opcion 2: Desde archivo
# tu_imagen = Image.open("ruta/a/tu/imagen.jpg").convert("RGB")

# Detectar
# detect_objects(tu_imagen, coco_model, conf_threshold=0.5)


### O usa la camara

Haz una foto a lo que tengas delante y detecta ahi. Los objetos de tu mesa que esten entre las 80 clases de COCO deberian salir.


In [ ]:
# Hacer una foto con la camara del portatil (solo funciona en Colab)
import sys
from base64 import b64decode

def hacer_foto(nombre="foto.jpg", calidad=0.92):
    """Abre la camara, espera a que pulses el boton y devuelve la ruta de la foto."""
    if "google.colab" not in sys.modules:
        raise RuntimeError("Esta celda solo funciona en Google Colab. En local, carga un fichero con Image.open().")

    from IPython.display import display, Javascript
    from google.colab.output import eval_js

    display(Javascript("""
        async function tomarFoto(calidad) {
          const div = document.createElement('div');
          const boton = document.createElement('button');
          boton.textContent = 'Hacer foto';
          boton.style.margin = '8px';
          const video = document.createElement('video');
          video.style.display = 'block';
          video.style.maxWidth = '480px';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(boton);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => boton.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', calidad);
        }
        """))
    datos = eval_js("tomarFoto({})".format(calidad))
    with open(nombre, "wb") as salida:
        salida.write(b64decode(datos.split(",")[1]))
    return nombre


In [ ]:
ruta = hacer_foto()
mi_foto = Image.open(ruta).convert("RGB")
detect_objects(mi_foto, coco_model, conf_threshold=0.4)


## Ejercicio Extra: Filtrar por clase

Modifica la funcion `detect_objects()` para que solo muestre detecciones de ciertas clases.

Por ejemplo: solo mostrar "car" y "truck", ignorando el resto.

**Pista**: Filtra las boxes antes de dibujarlas comprobando el nombre de la clase.

In [ ]:
# EJERCICIO EXTRA para casa: Deteccion filtrada por clases
def detect_only(image, allowed_classes, conf_threshold=0.5):
    """
    Detecta solo las clases especificadas.
    allowed_classes: lista de nombres de clases, ej: ['car', 'person']
    """
    # Tu codigo aqui...
    pass

# Prueba:
# detect_only(image_cars, allowed_classes=['car'], conf_threshold=0.5)


## Limitaciones de Modelos Supervisados

RF-DETR esta entrenado en COCO que tiene 80 categorias. Esto significa:

### ✅ Detecta bien:
- Objetos comunes: personas, coches, animales, muebles
- Objetos de la vida diaria

### ❌ NO detecta:
- Objetos industriales especificos (tornillos, tuercas, componentes electronicos)
- Objetos raros o muy especializados
- Defectos de fabricacion
- Cualquier cosa que no este en las 80 categorias COCO

**Solucion**: Fine Tuning

# Fine-Tuning: Entrenamiento de Modelo Custom

## ¿Por qué entrenar un modelo custom?

El modelo RF-DETR pre-entrenado en COCO es muy útil, pero tiene limitaciones:

- **COCO es genérico**: Solo detecta "person" sin distinguir roles o contextos específicos
- **Aplicaciones especializadas**: En deportes, seguridad, industria, etc., necesitamos más detalle
- **Clases específicas**: Queremos distinguir entre jugador, árbitro, pelota, etc.

### Ejemplo: Basketball Detection

Vamos a entrenar un modelo custom para detección en baloncesto usando un dataset muy pequeño de **Roboflow Universe**:

**Dataset**: [Basketball Players](https://universe.roboflow.com/roboflow-universe-projects/basketball-players-fy4c2)


## Proceso de Fine-Tuning

El **fine-tuning** consiste en:

1. **Partir del modelo pre-entrenado**: Aprovechamos el conocimiento de COCO (80 clases)
2. **Adaptar a nuestro dominio**: Re-entrenar las capas finales con nuestro dataset específico
3. **Obtener modelo especializado**: Mejor rendimiento en nuestra tarea concreta

**Ventajas**:
- ✅ Requiere menos datos que entrenar desde cero
- ✅ Converge más rápido (menos épocas)
- ✅ Mantiene capacidad de generalización base

**Requisitos**:
- Dataset anotado con bounding boxes (formato COCO, YOLO, etc.)
- Suficientes ejemplos por clase
- GPU para entrenamiento eficiente

In [ ]:
if "google.colab" in sys.modules: #Detectamos si estamos en google colab o en local
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/Modelos_Fundacionales/data/")
else:
    DATA_DIR = Path("./data/")
    
print(DATA_DIR)

### Antes de entrenar

El entrenamiento no viene en la instalacion basica de `rfdetr`: la celda de abajo instala `rfdetr[train,loggers]` la primera vez. Si aun asi falla con un `ModuleNotFoundError`, reinicia el entorno (Entorno de ejecucion, Reiniciar sesion) y vuelve a ejecutar desde la celda de imports.

Y comprueba que tienes GPU: en CPU, diez epocas no terminan en una clase.


In [ ]:
# El entrenamiento necesita dependencias que no trae la instalacion basica de rfdetr
try:
    import pytorch_lightning  # noqa: F401
except ImportError:
    %pip install -q "rfdetr[train,loggers]"

import torch
torch.cuda.empty_cache()  # Limpiar memoria GPU antes de entrenamiento

DATASET_DIR = DATA_DIR / "Basketball Players.v1-original_raw-images.coco"

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"No encuentro el dataset en {DATASET_DIR}.\n"
        "Descargalo en formato COCO desde Roboflow Universe y descomprimelo ahi:\n"
        "https://universe.roboflow.com/roboflow-universe-projects/basketball-players-fy4c2\n"
        "Debe quedar con las carpetas train/, valid/ y test/, cada una con su _annotations.coco.json."
    )

print("Dataset encontrado:", DATASET_DIR)
model_custom = RFDETRMedium()
model_custom.train(
    dataset_dir=str(DATASET_DIR),
    epochs=10  # Pocas epocas para que la clase no se eternice
)


## Carga del Modelo Entrenado y Comparación

Una vez completado el entrenamiento, tenemos un modelo especializado guardado en `output/checkpoint_best_total.pth`.

### Comparación COCO vs Custom

Vamos a comparar cómo funciona el modelo genérico COCO frente a nuestro modelo custom en la misma imagen:

In [ ]:
# Comparacion real: el mismo recorte visto por el modelo generico y por el entrenado
CHECKPOINT = Path("output/checkpoint_best_total.pth")

if not CHECKPOINT.exists():
    print("Todavia no hay checkpoint: ejecuta antes la celda de entrenamiento.")
else:
    model_custom_trained = RFDETRMedium(pretrain_weights=str(CHECKPOINT))

    image_path = DATASET_DIR / "test" / "youtube-15_jpg.rf.22a08fb220872e38ed069f28024679e0.jpg"
    custom_image = Image.open(image_path).convert("RGB")

    print("=== Modelo COCO (80 clases genericas) ===")
    detect_objects(custom_image, coco_model, conf_threshold=0.5)

    print("\n=== Modelo custom (clases del dataset de baloncesto) ===")
    print("Clases:", model_custom_trained.class_names)
    detect_objects(custom_image, model_custom_trained, conf_threshold=0.5)


## Novedad: la misma familia hace mas cosas

Desde este curso, `rfdetr` incluye segmentacion de instancias y puntos clave, ademas de deteccion. Cambia la clase y cambia lo que recibes, con la misma llamada.

Las dos estan en acceso temprano (`Preview`), asi que pueden cambiar de nombre o de firma.


In [ ]:
# Mascaras en vez de cajas
try:
    from rfdetr import RFDETRSegPreview

    seg_model = RFDETRSegPreview()
    seg = seg_model.predict(np.array(image_cars), threshold=0.5)
    print("Objetos segmentados:", len(seg.xyxy))
    print("Forma de las mascaras:", None if seg.mask is None else seg.mask.shape)

    if seg.mask is not None:
        plt.figure(figsize=(12, 8))
        plt.imshow(image_cars)
        plt.imshow(seg.mask.any(axis=0), alpha=0.45, cmap="cool")
        plt.title("Union de las mascaras")
        plt.axis("off")
        plt.show()
except ImportError as error:
    print("Tu version de rfdetr todavia no trae segmentacion:", error)


In [ ]:
# Puntos clave: 17 por persona, los mismos de COCO
try:
    from rfdetr import RFDETRKeypointPreview

    pose_model = RFDETRKeypointPreview()
    puntos = pose_model.predict(np.array(image_person), threshold=0.5)
    print("Personas detectadas:", len(puntos.xy))
    print("Forma de los puntos:", puntos.xy.shape)

    plt.figure(figsize=(12, 8))
    plt.imshow(image_person)
    for persona in puntos.xy:
        plt.scatter(persona[:, 0], persona[:, 1], s=40, c="yellow", edgecolors="black")
    plt.title("Puntos clave")
    plt.axis("off")
    plt.show()
except ImportError as error:
    print("Tu version de rfdetr todavia no trae puntos clave:", error)


## Resumen

En este notebook hemos aprendido:

✅ Que es la deteccion de objetos y para que sirve  
✅ Diferencia entre arquitecturas clasicas (YOLO) y modernas (RF-DETR)  
✅ Por que Transformers funcionan bien en vision  
✅ Usar RF-DETR para detectar objetos  
✅ Configurar thresholds de confianza  
✅ Limitaciones de modelos supervisados (numero de clases)  
✅ Contar objetos automaticamente   
✅ Descargar y usar datasets de Roboflow Universe  
✅ Proceso de fine-tuning: partir de modelo pre-entrenado y adaptar  
✅ Entrenar RF-DETR con dataset custom (Basketball Players)
✅ Ajustar el umbral con un deslizador
✅ Anotar resultados con supervision
✅ Detectar sobre una foto hecha con la camara
✅ Segmentacion y puntos clave con la misma familia de modelos
